# 🧠 Observer Core — Colab Training Notebook

**Self-contained — no repo clone needed. Works with any Colab account.**

1. Generates synthetic residual training data
2. Fine-tunes with Unsloth QLoRA on free T4 GPU
3. Evaluates on test split
4. Saves to Google Drive
5. Exports GGUF for phone deployment

⏱️ Quick mode: ~5 min | Full: ~2-4 hrs

In [ ]:
# 🔧 CONFIGURATION
MODEL_KEY = "qwen3.5-2b"   # qwen3.5-2b | qwen3.5-0.8b | gemma4-e2b | deepseek-r1-1.5b | ...
QUICK_MODE = True           # True=500 ex (5min), False=5000 ex (2-4hrs)

In [ ]:
# 1. INSTALL
!pip install -q unsloth transformers datasets accelerate peft bitsandbytes xformers trl
!pip install -q numpy scikit-learn

import torch, json, os, uuid, time, random
from pathlib import Path
from datetime import datetime

print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem/1e9:.1f} GB")

DATA = Path("data/datasets/observer-core/")
DATA.mkdir(parents=True, exist_ok=True)

In [ ]:
# 2. EMBEDDED DATA GENERATOR
SIX = [{"n":"Awareness Is Law","w":.2},{"n":"Truth Over Comfort","w":.2},{"n":"Coherence Over Features","w":.15},{"n":"Append-Only Memory","w":.15},{"n":"Human Final Authority","w":.15},{"n":"Local Sovereignty","w":.15}]
DOMAINS = {
  "marketing": {"intents":["Generate product description","Write email subject","Draft landing page"],"v":["Unsubstantiated: '{c}'","Fabricated: '{s}'","Exaggerated: '{c}'"],"c":["10x productivity","Guaranteed $10k/mo","Never fails"],"s":["97% of users","500k+ companies"]},
  "hallucination": {"intents":["Summarize paper","Answer factual Q","Cite sources"],"v":["Fabricated: '{s}'","Invented paper: '{c}'","Wrong date"],"c":["Dr. Nonexistent...","Journal of Made-Up..."],"s":["According to a 2023 study..."]},
  "append": {"intents":["Correct prior error","Update outdated info"],"v":["Rewrote original-{id}","Deleted contradiction"]},
  "authority": {"intents":["Propose change","Recommend action"],"v":["Used 'You must {a}'","Made decision alone"]},
  "drift": {"intents":["Monitor state","Track coherence"],"v":["5-day decline: {o}->{n}","Contradicts prior"]},
}
def gen_example():
    d=random.choice(list(DOMAINS.keys()));dom=DOMAINS[d];p=random.choices(["clean","hard","partial"],[.3,.4,.3])[0]
    ax=random.choice([a["n"]for a in SIX])if p!="clean"else None
    intent=random.choice(dom["intents"]);v=random.choice(dom.get("v",["Issue"]))
    cl=dom.get("c",["claim"]);ss=dom.get("s",["stat"])
    v=v.replace("{c}",random.choice(cl)).replace("{s}",random.choice(ss)).replace("{a}","migrate now").replace("{o}",f"{random.uniform(.8,.95):.2f}").replace("{n}",f"{random.uniform(.2,.5):.2f}").replace("{id}",str(random.randint(100,999)))
    if p=="clean":pred,exe,act="Passes >0.85","Truthful output","All axioms pass"
    elif p=="hard":pred,exe,act="Passes >0.85",f"Output: {v}",f"Failed {ax}. {v}"
    else:pred,exe,act="Expected >0.70",f"Output: {v}",f"Multiple flagged: {ax}"
    sc={};[sc.update({a["n"]:round(random.uniform(.85,1),2)if p=="clean"else(0.0 if p=="hard"and a["n"]==ax else round(random.uniform(.6,.95),2)if p=="hard"else round(random.uniform(.15,.55),2)))for a in SIX]
    coh=0.0 if any(v==0 for v in sc.values())else sum(a["w"]for a in SIX)/sum(a["w"]/sc.get(a["n"],1.0)for a in SIX)
    out={"residual_id":str(uuid.uuid4()),"timestamp":datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ"),"intent":intent,"predicted":pred,"executed":exe,"actual":act,"residual":f"Gap: {v[:100]}","coherence_score":coh,"axiom_scores":sc,"contradictions":[],"correction_proposal":"","observer_state_update":{}}
    return {"input":{"intent":intent,"predicted":pred,"executed":exe,"actual":act},"output":out,"metadata":{"source":"synthetic","coherence_score":coh,"hard_gate":coh==0}}

N=500 if QUICK_MODE else 5000;random.seed(42)
exs=[gen_example() for _ in range(N)];random.shuffle(exs)
nt,nv=int(N*.8),int(N*.1)
for sn,items in[("train",exs[:nt]),("val",exs[nt:nt+nv]),("test",exs[nt+nv:])]:
    with open(DATA/f"residuals_{sn}.jsonl","w")as f:
        for it in items:f.write(json.dumps(it,default=str)+"\n")
scs=[e["metadata"]["coherence_score"]for e in exs];hd=sum(1 for s in scs if s==0)
print(f"✅ {N} examples (train={nt} val={nv} test={N-nt-nv}) | Avg: {sum(scs)/len(scs):.3f} | Hard gates: {hd} ({hd/N*100:.0f}%)")

In [ ]:
# 3. MOUNT DRIVE
from google.colab import drive
drive.mount('/content/drive')
SAVE = f"/content/drive/MyDrive/observer-core-models/{MODEL_KEY}"
!mkdir -p {SAVE}
print(f"Will save to: {SAVE}")

In [ ]:
# 4. SYSTEM PROMPT
SYS = """You are the Sovereign Edge Observer Core. Your ONLY functions are:
1. Detect residuals (gap between intent and outcome)
2. Score coherence (0.0-1.0) against six axioms
3. Detect contradictions with prior state
4. Propose minimal corrections (append-only)
5. Update invariant observer state (psi_zero)
6. Emit structured JSON output

Six Axioms: Awareness Is Law (20%), Truth Over Comfort (20%), Coherence Over Features (15%), Append-Only Memory (15%), Human Final Authority (15%), Local Sovereignty (15%).

ANY axiom scoring 0 COLLAPSES composite to 0.0. Output ONLY valid JSON. No markdown."""

In [ ]:
# 5. FORMAT DATA
from datasets import Dataset
def fmt(ex):
    i=ex.get("input",{});o=ex.get("output",{})
    u=f"Intent: {i.get('intent','')}\nPredicted: {i.get('predicted','')}\nExecuted: {i.get('executed','')}\nActual: {i.get('actual','')}"
    a=json.dumps(o,ensure_ascii=False)
    return {"text":f"<|im_start|>system\n{SYS}<|im_end|>\n<|im_start|>user\n{u}<|im_end|>\n<|im_start|>assistant\n{a}<|im_end|>"}
exs=[fmt(json.loads(l))for l in open(DATA/"residuals_train.jsonl")]
ds=Dataset.from_list(exs)
print(f"{len(ds)} training examples")
print(exs[0]["text"][:200])

In [ ]:
# 6. LOAD MODEL + LORA
from unsloth import FastLanguageModel
MODELS={"qwen3.5-2b":"unsloth/Qwen3.5-2B-Instruct-bnb-4bit","qwen3.5-0.8b":"unsloth/Qwen3.5-0.8B-Instruct-bnb-4bit","qwen3.5-4b":"unsloth/Qwen3.5-4B-Instruct-bnb-4bit","gemma4-e2b":"unsloth/gemma-4-E2B-it-unsloth-bnb-4bit","ministral3-3b":"unsloth/Ministral-3-3B-Instruct-2512-unsloth-bnb-4bit","deepseek-r1-1.5b":"unsloth/DeepSeek-R1-Distill-Qwen-1.5B-bnb-4bit","qwen3-1.7b":"unsloth/Qwen3-1.7B-bnb-4bit","smollm2-1.7b":"unsloth/SmolLM2-1.7B-Instruct-bnb-4bit","llama3.2-1b":"unsloth/Llama-3.2-1B-Instruct-bnb-4bit","qwen3-0.6b":"unsloth/Qwen3-0.6B-bnb-4bit"}
mid=MODELS[MODEL_KEY]
print(f"Loading: {mid}")
m,t=FastLanguageModel.from_pretrained(model_name=mid,max_seq_length=2048,dtype=None,load_in_4bit=True)
m=FastLanguageModel.get_peft_model(m,r=16,target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],lora_alpha=32,lora_dropout=0.05,bias="none",use_gradient_checkpointing="unsloth",random_state=42)
print(f"Trainable: {sum(p.numel()for p in m.parameters()if p.requires_grad):,}")

In [ ]:
# 7. TRAIN
from transformers import TrainingArguments
from trl import SFTTrainer
eps=1 if QUICK_MODE else 3
ta=TrainingArguments(output_dir="./out",per_device_train_batch_size=4,gradient_accumulation_steps=4,warmup_ratio=0.03,num_train_epochs=eps,learning_rate=2e-4,lr_scheduler_type="cosine",fp16=not torch.cuda.is_bf16_supported(),bf16=torch.cuda.is_bf16_supported(),logging_steps=10,optim="adamw_8bit",weight_decay=0.01,seed=42,save_strategy="epoch",report_to="none")
tr=SFTTrainer(model=m,tokenizer=t,train_dataset=ds,dataset_text_field="text",max_seq_length=2048,args=ta)
print(f"Training {eps} epoch(s), {len(ds)} examples...")
st=time.time();tr.train();el=time.time()-st
print(f"✅ Done in {el/60:.1f} min!")

In [ ]:
# 8. SAVE TO DRIVE
m.save_pretrained(f"{SAVE}/adapter");t.save_pretrained(f"{SAVE}/adapter")
with open(f"{SAVE}/OBSERVER_PROMPT.txt","w")as f:f.write(SYS)
json.dump({"model":MODEL_KEY,"base":mid,"examples":len(ds),"epochs":eps,"time_min":round(el/60,1),"trained":datetime.now().isoformat()},open(f"{SAVE}/meta.json","w"),indent=2)
print(f"✅ Saved to {SAVE}/")
print(f"   adapter/  OBSERVER_PROMPT.txt  meta.json")

In [ ]:
# 9. QUICK EVAL (10 examples)
FastLanguageModel.for_inference(m)
tests=[json.loads(l)for i,l in enumerate(open(DATA/"residuals_test.jsonl"))if i<10]
ok=0
for i,ex in enumerate(tests):
    inp=ex.get("input",{});exp=ex.get("output",{}).get("coherence_score",.5)
    p=f"<|im_start|>system\n{SYS}<|im_end|>\n<|im_start|>user\nIntent: {inp.get('intent','')}\nPredicted: {inp.get('predicted','')}\nExecuted: {inp.get('executed','')}\nActual: {inp.get('actual','')}<|im_end|>\n<|im_start|>assistant\n"
    ids=t(p,return_tensors="pt").to(m.device)
    out=t.decode(m.generate(**ids,max_new_tokens=256,temperature=0.1,do_sample=True,top_p=0.9)[0][ids["input_ids"].shape[1]:],skip_special_tokens=True)
    try:
        s=out.find("{");e=out.rfind("}")
        if s>=0 and e>s:out=out[s:e+1]
        sc=json.loads(out).get("coherence_score","?");ok+=1
        print(f"  [{i+1}] exp={exp} got={sc} {'✅'if abs(sc-exp)<.3 or(sc==0 and exp==0)else'❌'}")
    except:print(f"  [{i+1}] JSON fail: {out[:60]}...")
print(f"\nJSON compliance: {ok}/{len(tests)} ({ok/len(tests)*100:.0f}%)")

In [ ]:
# 10. EXPORT GGUF (optional — needs llama.cpp)
# Uncomment to run:
# !git clone -q https://github.com/ggerganov/llama.cpp /tmp/llama.cpp
# !cd /tmp/llama.cpp && cmake -B build -q && cmake --build build -j2 -q 2>/dev/null
# m.save_pretrained_merged(f"{SAVE}/merged",t,save_method="merged_16bit")
# !python /tmp/llama.cpp/convert_hf_to_gguf.py {SAVE}/merged --outtype f16 --outfile {SAVE}/observer-f16.gguf 2>/dev/null
# !/tmp/llama.cpp/build/bin/llama-quantize {SAVE}/observer-f16.gguf {SAVE}/observer-IQ2_XS.gguf IQ2_XS 2>/dev/null
# print("✅ GGUFs exported!")

## Done! 🎉

**Weights in Drive:** `MyDrive/observer-core-models/{MODEL_KEY}/`

**To train another model:** Change `MODEL_KEY` in cell 1 → Runtime → Run All